# Multi-band fits

One physical model, several images. Here two bands of the same lens share **one**
`LensModel`; each band is its own `ImageData`, and `ProbModel` fits them jointly.
This is the pattern for any multi-dataset fit — the old `multiband` and `JointModel`
paths collapse into it. It also shows **two source planes** placed by
`deflection_ratio`, with light amplitudes solved by least squares (`use_lstsq=True`).

In [ ]:
%matplotlib inline
import numpy as np
import jax
from jax import numpy as jnp
import optax
import tensorflow_probability.substrates.jax as tfp
import matplotlib as mpl
from matplotlib import pyplot as plt
from corner import corner
tfd = tfp.distributions

import gigalens
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.scene_prob_model import ImageData, ProbModel
from gigalens.jax.inference import ModellingSequence
print('jax', jax.__version__, '| devices', jax.devices())

## Build the model

Each source gets **fresh** priors: the scene forbids reusing the *same* `tfd`
distribution object at two sites (use `shared()` to link deliberately), so a helper
keeps the two sources independent. The source planes are placed by `deflection_ratio`.

In [ ]:
epl = Component(EPL(), dict(
    theta_E=tfd.LogNormal(jnp.log(1.25), 0.25), gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
    e1=tfd.Normal(0, 0.1), e2=tfd.Normal(0, 0.1),
    center_x=tfd.Normal(0, 0.05), center_y=tfd.Normal(0, 0.05)))
shear = Component(Shear(), dict(gamma1=tfd.Normal(0, 0.05), gamma2=tfd.Normal(0, 0.05)))
# fresh distributions per source: the scene forbids reusing the same tfd object at two sites
# (that is what shared(...) is for); each source's priors are independent here.
def src_priors():
    return dict(R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15), n_sersic=tfd.Uniform(0.5, 4),
                e1=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5), e2=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.25), center_y=tfd.Normal(0, 0.25))
source_1 = Component(SersicEllipse(use_lstsq=True), src_priors())
source_2 = Component(SersicEllipse(use_lstsq=True), src_priors())

model = LensModel([
    Plane(mass=[epl, shear]),
    Plane(deflection_ratio=1.0, light=[source_1]),
    Plane(deflection_ratio=1.5, light=[source_2]),
])
print('free parameters:', model.num_free_params)

## Data: one `ImageData` per band

Every band is its own `ImageData` (all sharing this `sim_config`), and `ProbModel`
takes the **list**. `sees="all"` routes every light component into every band; finer
routing would name specific components instead of `"all"`.

:::{admonition} `mode='lstsq'` with `use_lstsq=True`
:class: tip
With linear light amplitudes, run the prob model in `mode='lstsq'`: at each step the
source amplitudes are solved by least squares rather than sampled.
:::

In [ ]:
root = gigalens.__path__[0]
kernel = np.load(f'{root}/assets/psf.npy').astype(np.float32)
observed_imgs = np.load(f'{root}/assets/multiband_demo.npy')
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=1, kernel=kernel,
                             likelihood_precision='float32')
background_rms, exp_time = 0.2, 100

datasets = [ImageData(observed_imgs[b], sim_config, background_rms=background_rms,
                      exp_time=exp_time, sees='all') for b in range(2)]
prob = ProbModel(model, datasets, mode='lstsq')
seq = ModellingSequence(prob)

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for b in range(2):
    im = ax[b].imshow(observed_imgs[b], vmin=0, vmax=5); ax[b].set_title(f'Band {b}')
    plt.colorbar(im, ax=ax[b])
plt.show()

## Inference: MAP → SVI → HMC

Joint fitting across both bands runs through the same `prob.log_prob(z)`.

In [ ]:
names = list(model.z_param_names)
def to_constrained(z_rows):
    xb = model.bijector.forward(jnp.asarray(z_rows).reshape(-1, len(names)))
    return np.stack([np.asarray(xb[n]).reshape(-1) for n in names], axis=1)

opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
best_z, best_lp, _ = seq.MAP(opt, n_samples=512, num_steps=500, seed=0, output_type='best')
best_z = np.asarray(jax.device_get(best_z))
print('MAP log-post: %.4g' % float(best_lp))

In [ ]:
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = seq.SVI(best_z, opt, n_vi=512, num_steps=2500, seed=0)
plt.plot(np.asarray(loss_hist).reshape(-1)); plt.xlabel('step'); plt.ylabel('-ELBO'); plt.title('SVI loss'); plt.show()

In [ ]:
# Rebuild the SVI surrogate from host arrays: under JAX 0.10 the sharded qz trips a
# mesh (Manual vs Explicit) clash inside HMC's pmapped sampler. device_get strips the
# sharding tag while keeping the SVI-learned mean + covariance.
qz = tfd.MultivariateNormalFullCovariance(
    loc=np.asarray(jax.device_get(qz.mean())),
    covariance_matrix=np.asarray(jax.device_get(qz.covariance())))
samples = seq.HMC(qz, n_hmc=16, num_burnin_steps=500, num_results=1000)
rhat = np.asarray(tfp.mcmc.potential_scale_reduction(samples, independent_chain_ndims=2))
ess = np.asarray(tfp.mcmc.effective_sample_size(samples, cross_chain_dims=[1, 2]))
print('max R-hat: %.3f | min ESS: %.0f' % (np.nanmax(rhat), np.nanmin(ess)))

## Best-fit reconstruction and residuals (per band)

Per-band reconstructions come from `prob.simulators[b]` — one `SceneSimulator` per
dataset, in dataset order.

In [ ]:
n_params = samples.shape[-1]
post = to_constrained(np.asarray(samples).reshape(-1, n_params))
best_fit = np.median(post, axis=0)
best_unique = {n: jnp.asarray(best_fit[i]) for i, n in enumerate(names)}
params = model.to_params(best_unique)

fig, ax = plt.subplots(2, 3, figsize=(12, 7))
for b in range(2):
    sim_b = prob.simulators[b]
    recon = np.asarray(sim_b.lstsq_simulate(params, datasets[b].image, datasets[b].error_map, datasets[b].mask))
    resid = (observed_imgs[b] - recon) / np.asarray(datasets[b].error_map)
    for col, (img, ttl, kw) in enumerate([
            (observed_imgs[b], f'Band {b} observed', dict(vmin=0, vmax=5)),
            (recon, f'Band {b} best fit', dict(vmin=0, vmax=5)),
            (resid, f'Band {b} residual', dict(cmap='bwr', vmin=-6, vmax=6))]):
        im = ax[b, col].imshow(img, **kw); ax[b, col].axis('off'); ax[b, col].set_title(ttl)
        plt.colorbar(im, ax=ax[b, col])
plt.tight_layout(); plt.show()

## Posterior: lens mass parameters

Map samples to physical values with `model.bijector.forward(...)`; column order is
`list(model.z_param_names)`, and parameters are addressed by path strings like
`planes/0/mass/0/theta_E`.

In [ ]:
mass_paths = ['planes/0/mass/0/theta_E', 'planes/0/mass/0/gamma', 'planes/0/mass/0/e1',
              'planes/0/mass/0/e2', 'planes/0/mass/0/center_x', 'planes/0/mass/0/center_y',
              'planes/0/mass/1/gamma1', 'planes/0/mass/1/gamma2']
mass_labels = [r'$\theta_E$', r'$\gamma$', r'$e_1$', r'$e_2$', r'$x_{lens}$', r'$y_{lens}$',
               r'$\gamma_{1,\rm ext}$', r'$\gamma_{2,\rm ext}$']
truth_mass = [1.1, 2.0, 0.1, 0.1, 0.1, 0.0, -0.01, 0.03]
idx = [names.index(p) for p in mass_paths]
fig = corner(post[:, idx], labels=mass_labels, truths=truth_mass, show_titles=True, title_fmt='.3f')
fig.suptitle('Lens mass parameters'); plt.show()

## Note — forward-mode multi-band

`mode="lstsq"` above solves each band's amplitudes independently. To instead **sample** amplitudes
(forward mode) across bands, you cannot reuse one light Component for both bands — it would render
identical flux in every filter, which the scene API rejects (`_validate_forward_flux_sharing`).
Give each band its own light Component and tie only the geometry with `shared(...)`:

```python
from gigalens.jax.scene import shared
Rs = shared(tfd.LogNormal(jnp.log(0.25), 0.15))   # one shared radius, ...
cx = shared(tfd.Normal(0, 0.25)); cy = shared(tfd.Normal(0, 0.25))
src_b0 = Component(SersicEllipse(use_lstsq=False), dict(..., R_sersic=Rs, center_x=cx, center_y=cy,
                                                        Ie=tfd.LogNormal(jnp.log(150.), 0.5)))
src_b1 = Component(SersicEllipse(use_lstsq=False), dict(..., R_sersic=Rs, center_x=cx, center_y=cy,
                                                        Ie=tfd.LogNormal(jnp.log(150.), 0.5)))
# band 0 sees src_b0, band 1 sees src_b1 (per-band Ie, shared geometry), mode="forward"
```